# Service Hours dropdown chart

Is there a way to organize chart is to put all 3 day_types for 1 date in one chart
and let dropdown move from 1 month to another?

In [1]:
import altair as alt
import gcsfs
import google.auth
import pandas as pd

import _new_operator_report_utils as utils
import _portfolio_charts

from update_vars import (
    DIGEST_DICT, PROCESSED_GCS, 
    abbrev_month, readable_dict, analysis_month
)

In [2]:
analysis_name = "Alameda-Contra Costa Transit District"

In [3]:
operator_hourly_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.hourly_day_type_summary}_{abbrev_month}.parquet"

operator_hourly_summary_df = pd.read_parquet(
    operator_hourly_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[[("Analysis Name", "==", analysis_name),
    ("Departure Hour", "<=", 24)]]
).reset_index(drop=True)

In [4]:
from IPython.display import HTML
# Set drop down menu to be on the upper right for the charts
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [5]:
date_list = operator_hourly_summary_df["Date"].unique()

date_dropdown = alt.binding_select(
    options=date_list,
    name="Dates: ",
)
xcol_param = alt.selection_point(
    fields=["Date"], value=date_list[0], bind=date_dropdown
)

In [6]:
date_list

array(['07-2025', '08-2025', '09-2025', '10-2025', '11-2025', '12-2025',
       '01-2026', '02-2026', '03-2026', '04-2026', '05-2026', '06-2026',
       '07-2026'], dtype=object)

In [7]:
# want to sort reverse, most recent date first
datetime_list = pd.to_datetime(operator_hourly_summary_df.Date.unique(), format="%m-%Y")
new_date_list = [i.strftime("%m-%Y") for i in reversed(datetime_list)]
new_date_list

['07-2026',
 '06-2026',
 '05-2026',
 '04-2026',
 '03-2026',
 '02-2026',
 '01-2026',
 '12-2025',
 '11-2025',
 '10-2025',
 '09-2025',
 '08-2025',
 '07-2025']

In [8]:
chart_dict = readable_dict.hourly_summary

In [9]:
# utils.create_hourly_summary
def create_hourly_summary_new(df):

    selection = alt.selection_point(fields=["Day Type"], bind = "legend")
    
    chart = (
        alt.Chart(df)
        .mark_line(size=3)
        .encode(
            x=alt.X(
                "Departure Hour",
                title="Departure Hour",
                axis=alt.Axis(
                    labelAngle=-45,
                ),
            ),
            y=alt.Y(
                "N Trips",
                title="N Trips",
            ),
            color = alt.Color(
                "Day Type:N", 
                scale=alt.Scale(
                    domain=["Weekday", "Saturday", "Sunday"], 
                    #range=[*chart_dict.colors]
                    range=["#dd217d", "#fc5c04", "#ccbb44"] # set these colors to be different than background
                )
            ),
            opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
            tooltip = ["Analysis Name", "Date", "Day Type", "Departure Hour", "N Trips"]
        ).transform_filter(xcol_param)
    ).add_params(selection, xcol_param)

    return chart

In [10]:
bg = _portfolio_charts.create_bg_service_chart()

In [11]:
bg

alt.Chart(...)

In [12]:
chart = create_hourly_summary_new(operator_hourly_summary_df)

#https://github.com/vega/altair/issues/772
combined_chart = (bg + chart).properties(   
    resolve=alt.Resolve(
        scale=alt.LegendResolveMap(color=alt.ResolveMode("independent")),
    )
).interactive()

In [13]:
# Adjust title, now that all 3 day types are combined
_portfolio_charts.configure_chart(
    combined_chart,
    width=400,
    height=250,
    title=f"{chart_dict.title}",
    subtitle=chart_dict.subtitle,
)

alt.LayerChart(...)

In [14]:
# Make changes in utils and check
utils.create_hourly_summary(operator_hourly_summary_df)

alt.LayerChart(...)